# AssistIQ — Phase 1: Intent Classification Baseline & Evaluation

This notebook provides a reproducible demonstration of **Phase 1: Intent Classification** for **AssistIQ** (SpotifyCares Twitter Customer Support).

### Objectives
1. **Inspect & Validate Golden Set**: Validate the 200-example golden set (`dataset/golden_set.csv`).
2. **Train/Test Split**: Establish a strictly un-leaked, stratified 75% train / 25% test split (`random_state=2026`).
3. **Baseline 1 (Majority Class)**: Predict the most frequent class in the training set.
4. **Baseline 2 (TF-IDF + Logistic Regression)**: Train a balanced n-gram linear model.
5. **Proposed Model (TF-IDF + LinearSVC)**: Train a linear max-margin classifier with balanced class weights.
6. **Evaluation & Failure Analysis**: Compare Accuracy, Macro F1, Weighted F1, Confusion Matrices, and inspect misclassified test examples.

### 1. Import Dependencies & Load Golden Set
We use the modular functions defined in `backend/src/intent/`.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in path
module_path = os.path.abspath(os.path.join('..'))
backend_path = os.path.abspath(os.path.join('..', 'backend'))
for p in [module_path, backend_path]:
    if p not in sys.path:
        sys.path.append(p)

from src.intent.data import INTENT_TAXONOMY, INTENT_DEFINITIONS, validate_golden_set, load_and_split_data
from src.intent.models import build_majority_baseline, build_tfidf_logistic_regression, build_proposed_model

golden_csv = "../dataset/golden_set.csv" if os.path.exists("../dataset/golden_set.csv") else "dataset/golden_set.csv"
df = validate_golden_set(golden_csv)
df.head()

### 2. Class Distribution Across the 11-Intent Taxonomy

In [ ]:
dist = df['intent'].value_counts()
plt.figure(figsize=(10, 5))
dist.plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('Golden Set Class Distribution (N=200)', fontsize=14, fontweight='bold')
plt.xlabel('Number of Examples')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 3. Stratified Train / Test Split (75% / 25%, random_state=2026)
The test set (50 examples) is isolated completely before any feature extraction or model training.

In [ ]:
X_train, X_test, y_train, y_test, train_df, test_df = load_and_split_data(
    filepath=golden_csv,
    test_size=0.25,
    random_state=2026
)

split_summary = pd.DataFrame({
    'Train (75%)': train_df['intent'].value_counts(),
    'Test (25%)': test_df['intent'].value_counts()
}).fillna(0).astype(int)
split_summary['Total'] = split_summary['Train (75%)'] + split_summary['Test (25%)']
split_summary

### 4. Model Training & Comparative Evaluation
We train and evaluate all three classifiers on the unseen test set.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

models = {
    'majority_baseline': build_majority_baseline(),
    'tfidf_logistic_regression': build_tfidf_logistic_regression(random_state=2026),
    'proposed_model (LinearSVC)': build_proposed_model(random_state=2026)
}

metrics_list = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    metrics_list.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Macro F1': round(macro_f1, 4),
        'Weighted F1': round(weighted_f1, 4)
    })

comparison_df = pd.DataFrame(metrics_list)
comparison_df

### 5. Inspecting Stored Evaluation Results
Let's load the generated metrics, predictions, and misclassified errors.

In [ ]:
results_path = "../evaluation/results/intent_results.csv" if os.path.exists("../evaluation/results/intent_results.csv") else "evaluation/results/intent_results.csv"
summary_df = pd.read_csv(results_path)
print("=== Model Comparison ===")
display(summary_df)

per_class_path = "../evaluation/results/intent_per_class_results.csv" if os.path.exists("../evaluation/results/intent_per_class_results.csv") else "evaluation/results/intent_per_class_results.csv"
per_class_df = pd.read_csv(per_class_path)
print("\n=== Proposed Model Per-Class Metrics ===")
display(per_class_df[per_class_df['model'] == 'proposed_model'])

### 6. Error Analysis: Misclassified Test Examples
Inspect misclassified tweets from the Logistic Regression baseline alongside their top-3 predicted probabilities.

In [ ]:
errors_path = "../evaluation/results/intent_errors.csv" if os.path.exists("../evaluation/results/intent_errors.csv") else "evaluation/results/intent_errors.csv"
errors_df = pd.read_csv(errors_path)
print(f"Total Misclassified Test Examples: {len(errors_df)} / 50")

# Display first 5 errors with context
for idx, row in errors_df.head(5).iterrows():
    print(f"[Tweet ID: {row.tweet_id}]")
    print(f"  Text:        {row.text}")
    print(f"  True Intent: {row.true_intent}")
    print(f"  Predicted:   {row.predicted_intent} (prob={row.max_class_probability})")
    print(f"  Top-2:       {row.top_2_intent} (prob={row.top_2_prob})")
    print(f"  Top-3:       {row.top_3_intent} (prob={row.top_3_prob})")
    print("-" * 80)